# 推理服务与量化补充线 · 第 8/8 课：Serving Benchmark、SLO Goodput 与容量设计

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现 nearest-rank 百分位和 SLO goodput，设计能区分 TTFT、ITL、E2E 与吞吐的压测。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/lesson12` 已讲 MFU/HFU；在线服务评价的是请求分布下的延迟、吞吐、goodput、错误率和成本。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

压测产生到达过程与请求长度分布，记录 TTFT、逐 token ITL/TPOT、E2E、输出 token/s。SLO goodput 只统计满足全部服务目标的请求率。

### 数据与控制如何流动

先固定模型、硬件、请求长度与到达分布，再按 open-loop 速率逐级施压；每个窗口将成功、超时、拒绝和各延迟分位统一入账，找到满足全部 SLO 的最大持续速率。

### 正确性条件与常见误区

闭环客户端会在服务变慢时自动降低发送速率，造成 coordinated omission；百分位必须明确算法和样本窗口，错误/超时请求不能从分母消失。

### 性能、成本与工程取舍

追求最大 token/s 往往牺牲 p99；合理容量点应在目标流量、突发和失败条件下满足 SLO，并保留扩缩容/故障余量。

## 具体演示

10 个请求中 8 个满足 TTFT≤1s、p-token ITL≤50ms、E2E≤10s；60 秒窗口 SLO goodput=8/60 req/s，而非总到达 10/60。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐 nearest-rank p 分位与 SLO goodput；超时请求也留在 records 中。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
import math

def percentile_nearest_rank(values, p):
    if not values or not 0 < p <= 100:
        raise ValueError("invalid percentile")
    ordered = sorted(values)
    return ordered[math.ceil(p / 100 * len(ordered)) - 1]

def slo_goodput(records, window_s, ttft_slo, e2e_slo):
    if window_s <= 0:
        raise ValueError("window must be positive")
    # 每条 record = (ttft, e2e, success)。
    good = sum(1 for ttft, e2e, success in records
               if success and ttft <= ttft_slo and e2e <= e2e_slo)
    # TODO：返回满足 SLO 的请求率，而不是成功比例。
    return ______

records = [(0.5, 5, True), (1.2, 6, True), (0.8, 20, False)]
assert percentile_nearest_rank([1, 2, 3, 4], 95) == 4
assert slo_goodput(records, 60, 1.0, 10.0) == 1/60


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么只报平均 E2E 和总 token/s 无法判断服务是否可用？

**你的答案：**


### Q2

闭环压测为什么会低估过载时延迟？

**你的答案：**


### Q3

量化后 token/s 提升 30%，但 SLO goodput 不变，可能是什么原因？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
import math

def percentile_nearest_rank(values, p):
    if not values or not 0 < p <= 100:
        raise ValueError("invalid percentile")
    ordered = sorted(values)
    return ordered[math.ceil(p / 100 * len(ordered)) - 1]

def slo_goodput(records, window_s, ttft_slo, e2e_slo):
    if window_s <= 0:
        raise ValueError("window must be positive")
    good = sum(1 for ttft, e2e, success in records
               if success and ttft <= ttft_slo and e2e <= e2e_slo)
    return good / window_s

records = [(0.5, 5, True), (1.2, 6, True), (0.8, 20, False)]
assert percentile_nearest_rank([1, 2, 3, 4], 95) == 4
assert slo_goodput(records, 60, 1.0, 10.0) == 1/60


### Q1 参考答案

平均值隐藏排队和长请求尾部，token/s 可能由少数长输出贡献；用户体验还取决于 TTFT、ITL、错误/超时和请求长度分层。需要 p50/p95/p99 与 SLO goodput。

### Q2 参考答案

客户端等响应后才发下一个请求，服务变慢时发送率随之下降，未模拟真实外部到达队列；最糟等待发生在客户端而未被记录。应使用受控 open-loop 到达并记录排队/拒绝。

### Q3 参考答案

瓶颈可能在排队、tokenization、网络、TTFT prefill、KV 容量或尾延迟；也可能量化引入质量回退导致业务重试。应分解阶段指标并按请求长度/并发比较。

## 参考资料

- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)
- [torchao quantized inference](https://docs.pytorch.org/ao/stable/workflows/inference.html)

API 与平台能力会演进；部署前应按目标版本重新核对。